# W08 LSTM Battery Degradation Follow-Up

**Supervisor question:** Analyze the LSTM Autoencoder's performance on `accelerated_battery_degradation` before finalizing the capstone.

This notebook is the reproducible companion to `W08_LSTM_Battery_Degradation_Analysis.md`.

The saved `results/` files were generated by reusing the final Week 4 implementation for the same 10 split seeds. The follow-up adds threshold sensitivity, event-level detection, feature-level reconstruction diagnostics, battery-focused scoring, and epoch sensitivity.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

HERE = Path.cwd()
RESULTS = HERE / "results"
FIGURES = HERE / "figures"

required = [
    RESULTS / "lstm_battery_method_comparison.csv",
    RESULTS / "lstm_battery_reproduction_check.csv",
    RESULTS / "lstm_battery_event_diagnostics.csv",
    RESULTS / "lstm_battery_threshold_sensitivity_summary.csv",
    RESULTS / "lstm_battery_feature_reconstruction_summary.csv",
    RESULTS / "lstm_battery_focused_score_summary.csv",
    RESULTS / "lstm_battery_epoch_sensitivity_summary.csv",
    RESULTS / "summary_metrics.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("\n".join(missing))

print("PASS: committed follow-up outputs found.")


## Optional full reproduction

Run the next cell only when the repository contains the Week 3 five-unit telemetry sample and the final Week 4 notebook/results. The driver reuses the Week 4 implementation cells and regenerates the diagnostic result tables.


In [ ]:
# Full reproduction from the repository root.
# This can take a few minutes on a laptop CPU.
#
# !bash run_followup.sh


## 1. Cross-method battery-degradation comparison

In [ ]:
comparison = pd.read_csv(RESULTS / "lstm_battery_method_comparison.csv")
display(comparison.round(4))
display(Image(filename=str(FIGURES / "01_method_auroc_comparison.png")))


## 2. Exact reproduction of the final Week 4 LSTM result

The additional diagnostics are only used after verifying that the original battery metrics reproduce.


In [ ]:
check = pd.read_csv(RESULTS / "lstm_battery_reproduction_check.csv")
display(check.round(10))
print("Max AUROC absolute difference:", check["auroc_abs_diff"].max())


## 3. Split-level stability

In [ ]:
display(Image(filename=str(FIGURES / "02_lstm_split_auroc.png")))


## 4. Event-level behavior

The event table checks whether each injected battery event crosses the original 97th-percentile reconstruction threshold at least once.


In [ ]:
events = pd.read_csv(RESULTS / "lstm_battery_event_diagnostics.csv")
display(
    events[
        [
            "split_seed", "fault_id", "robot_id", "duration_s",
            "severity", "event_detected", "flagged_fault_windows",
            "fault_window_recall", "detection_latency_s",
        ]
    ]
)
print("Detected events:", int(events["event_detected"].sum()), "/", len(events))
display(Image(filename=str(FIGURES / "06_event_detection.png")))


## 5. Threshold sensitivity

A lower threshold should improve recall if calibration is the main issue. The diagnostic also tracks clean test-window flag rate, because a threshold that recovers faults by flooding the queue with clean alerts is not operationally useful.


In [ ]:
threshold = pd.read_csv(
    RESULTS / "lstm_battery_threshold_sensitivity_summary.csv"
)
display(threshold.round(4))
display(Image(filename=str(FIGURES / "03_threshold_sensitivity.png")))


## 6. Feature-level reconstruction diagnosis

The original LSTM score averages reconstruction MSE across all 17 inputs. The feature-level table checks whether battery-specific reconstruction channels contain more signal than the global score.


In [ ]:
feature = pd.read_csv(
    RESULTS / "lstm_battery_feature_reconstruction_summary.csv"
)
display(feature.head(12).round(4))

focused = pd.read_csv(
    RESULTS / "lstm_battery_focused_score_summary.csv"
)
display(focused.round(4))

display(Image(filename=str(FIGURES / "04_feature_reconstruction_auroc.png")))


## 7. Epoch sensitivity

The Week 4 model used one epoch for laptop portability. This sensitivity check holds the architecture and data constant and compares 1, 5, and 10 epochs.


In [ ]:
epochs = pd.read_csv(
    RESULTS / "lstm_battery_epoch_sensitivity_summary.csv"
)
display(epochs.round(4))
display(Image(filename=str(FIGURES / "05_epoch_sensitivity.png")))


## 8. Final interpretation

The configured LSTM Autoencoder is not competitive on accelerated battery degradation. Looser thresholds and additional epochs do not fix the result. Battery-specific reconstruction channels contain more signal than the global equal-weight MSE, so the most promising future sequence-model experiment is a battery-weighted score with longer-horizon health features and a larger clean training history.

The capstone recommendation remains LOF as the primary accuracy-maximizing anomaly detector for the current benchmark.
